# 笔记本 05 — 均值回归与配对交易

**阶段 2 · 策略模块 (2 / 4)**

---

## 🎯 学习目标

| # | 目标 |
|---|------|
| 1 | 理解均值回归假说：*跌多了必然反弹* |
| 2 | 从零构建布林带 + RSI 信号强度指标 |
| 3 | 理解协整检验和增广 Dickey-Fuller (ADF) 检验 |
| 4 | 计算 OLS 对冲比率和均值回归半衰期 |
| 5 | 构建基于 z-score 的入场 / 出场规则 |
| 6 | 将手写代码与 `evaluate_mean_reversion_signal()` 和 `find_cointegrated_pairs()` 对比 |

### 前置要求
- NB02（RSI、布林带）
- NB04（动量 — *相反*的策略）

In [ ]:
# ── 初始化 ─────────────────────────────────────────────────
import sys, pathlib, warnings
warnings.filterwarnings("ignore")
ROOT = str(pathlib.Path.cwd().resolve().parents[1])
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from scipy import stats

plt.rcParams.update({"figure.figsize": (14, 5), "axes.grid": True})
print("✅ 导入完毕  |  项目根目录:", ROOT)

---
## 1 · 均值回归假说

**动量**策略押注赢家继续赢，而**均值回归**则押注相反：跌过头的资产会反弹。

两个互补的入场信号：

| 信号 | 触发条件 | 直觉 |
|------|----------|------|
| **RSI 超卖** | RSI ≤ 30 | 卖压耗尽 |
| **布林带突破** | 价格 ≤ 下轨 | 价格低于均值 2σ 以上 |

我们的生产代码将两者合并为一个**信号强度** ∈ [0, 1]：

$$
\text{RSI\_signal} = \text{clip}\!\left(\frac{\text{RSI}_{\text{oversold}} - \text{RSI}}{\text{RSI}_{\text{oversold}}},\; 0,\; 1\right)
$$
$$
\text{BB\_signal} = \text{clip}\!\left(\frac{\text{下轨} - P}{P} \times 20,\; 0,\; 1\right)
$$
$$
\text{strength} = \max(\text{RSI\_signal},\; \text{BB\_signal})
$$

强度为 0 表示
；1 表示非常强的超卖读数。

---
## 2 · 合成数据

In [ ]:
np.random.seed(42)
DAYS = 120
dates = pd.date_range(end=pd.Timestamp.now().normalize(), periods=DAYS, freq="D")

# 带有崩盘和恢复的资产（均值回归的理想标的）
crash_start, crash_end = 60, 80
base_drift = np.random.normal(0.001, 0.02, DAYS)
base_drift[crash_start:crash_end] = np.random.normal(-0.03, 0.02, crash_end - crash_start)
base_drift[crash_end:crash_end+10] = np.random.normal(0.02, 0.015, 10)

eth_prices = 3500 * np.exp(np.cumsum(base_drift))
eth_volumes = np.random.uniform(10e6, 60e6, DAYS)

prices = pd.Series(eth_prices, index=dates, name="ETHUSDT")
quote_volumes = pd.Series(eth_volumes, index=dates, name="ETHUSDT")

print(f"价格范围: {prices.min():.0f} – {prices.max():.0f}")
prices.plot(title="ETHUSDT – 合成崩盘与恢复", ylabel="价格 (USD)")
plt.show()

---
## 3 · 构建均值回归指标框架

### 3.1  布林带 + RSI

In [ ]:
# ── RSI（来自 NB02）────────────────────────────────────────
def calculate_rsi(prices: pd.Series, period: int = 14) -> pd.Series:
    delta = prices.astype(float).diff()
    gains = delta.clip(lower=0.0)
    losses = (-delta).clip(lower=0.0)
    avg_gain = gains.ewm(alpha=1 / period, adjust=False, min_periods=period).mean()
    avg_loss = losses.ewm(alpha=1 / period, adjust=False, min_periods=period).mean()
    avg_loss_safe = avg_loss.mask(avg_loss == 0.0)
    rs = avg_gain / avg_loss_safe
    rsi = 100.0 - (100.0 / (1.0 + rs))
    rsi = rsi.mask((avg_loss == 0.0) & (avg_gain > 0.0), 100.0)
    rsi = rsi.mask((avg_gain == 0.0) & (avg_loss > 0.0), 0.0)
    rsi = rsi.mask((avg_gain == 0.0) & (avg_loss == 0.0), 50.0)
    return rsi.astype(float)

# ── 布林带 ─────────────────────────────────────────────────
BB_PERIOD = 20
BB_STD    = 2.0
RSI_OVERSOLD = 30.0

ma = prices.rolling(BB_PERIOD).mean()
std = prices.rolling(BB_PERIOD).std(ddof=0)
upper_band = ma + BB_STD * std
lower_band = ma - BB_STD * std
rsi = calculate_rsi(prices)

# 信号强度
rsi_signal = ((RSI_OVERSOLD - rsi) / RSI_OVERSOLD).clip(lower=0.0, upper=1.0)
bb_signal  = (((lower_band - prices) / prices) * 20.0).clip(lower=0.0, upper=1.0)
strength   = pd.concat([rsi_signal, bb_signal], axis=1).max(axis=1)

print(f"最大信号强度: {strength.max():.3f}")
print(f"信号强度 > 0 的天数: {(strength > 0).sum()}")

### 3.2  信号可视化

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

# ── 面板 1: 价格 + 布林带 ──
ax = axes[0]
ax.plot(prices.index, prices, label="收盘价", lw=1.5)
ax.plot(ma.index, ma, label="MA(20)", ls="--", lw=1)
ax.fill_between(prices.index, lower_band, upper_band, alpha=0.15, color="blue", label="布林带")
# 标记超卖入场点
entry_mask = strength > 0
ax.scatter(prices.index[entry_mask], prices[entry_mask],
           color="red", s=20, zorder=5, label="信号开启")
ax.set_ylabel("价格")
ax.legend(loc="upper left", fontsize=8)
ax.set_title("价格与布林带 & 均值回归信号")

# ── 面板 2: RSI ──
ax2 = axes[1]
ax2.plot(rsi.index, rsi, color="purple", lw=1.2)
ax2.axhline(RSI_OVERSOLD, color="green", ls="--", lw=1, label="超卖 = 30")
ax2.axhline(70, color="red", ls="--", lw=1, label="超买 = 70")
ax2.fill_between(rsi.index, 0, RSI_OVERSOLD, alpha=0.1, color="green")
ax2.set_ylabel("RSI")
ax2.set_ylim(0, 100)
ax2.legend(loc="upper left", fontsize=8)

# ── 面板 3: 信号强度 ──
ax3 = axes[2]
ax3.fill_between(strength.index, 0, strength, color="orange", alpha=0.6)
ax3.set_ylabel("信号强度")
ax3.set_ylim(0, 1.1)
ax3.xaxis.set_major_formatter(mdates.DateFormatter("%m-%d"))

plt.tight_layout()
plt.show()

---
## 4 · 生产代码：`evaluate_mean_reversion_signal()`

将我们的手写指标与生产函数进行对比。

In [ ]:
from bot.signals.mean_reversion import (
    build_mean_reversion_frame,
    evaluate_mean_reversion_signal,
    MeanReversionSignal,
)

# 构建完整指标框架
prod_frame = build_mean_reversion_frame(
    prices, quote_volumes,
    rsi_period=14,
    rsi_oversold=30.0,
    bollinger_period=20,
    bollinger_std=2.0,
    volume_window=24,
)

print("生产指标框架列:", prod_frame.columns.tolist())
prod_frame.tail(5)

In [ ]:
# 获取最新信号
signal = evaluate_mean_reversion_signal(
    prices, quote_volumes,
    rsi_period=14,
    rsi_oversold=30.0,
    bollinger_period=20,
    bollinger_std=2.0,
    min_volume_usd=10_000_000.0,
)

if signal is not None:
    print(f"检测到信号！")
    print(f"  强度        = {signal.strength:.3f}")
    print(f"  价格        = {signal.price:.2f}")
    print(f"  均线        = {signal.moving_average:.2f}")
    print(f"  下轨        = {signal.lower_band:.2f}")
    print(f"  RSI         = {signal.rsi:.1f}")
    print(f"  24h成交量   = {signal.volume_24h:,.0f}")
else:
    print("最新 K 线无均值回归信号（价格可能未超卖）。")
    print(f"  最新 RSI: {rsi.iloc[-1]:.1f}  (阈值: {RSI_OVERSOLD})")
    print(f"  最新价格: {prices.iloc[-1]:.2f}  下轨: {lower_band.iloc[-1]:.2f}")

---
## 5 · 配对交易：理论基础

### 5.1  什么是协整？

两个价格序列 $P_A$ 和 $P_B$ 是**协整**的，当且仅当存在常数 $\beta$（对冲比率）使得价差：

$$
S_t = P_{A,t} - \beta \cdot P_{B,t}
$$

是**平稳**的 — 它围绕一个常数均值波动并回归到它。这比相关性*更强*：相关的资产一起波动，但协整的资产是*绑定在一起*的。

### 5.2  交易逻辑

将价差标准化为 **z-score**：

$$
z_t = \frac{S_t - \bar{S}}{\sigma_S}
$$

| 条件 | 操作 |
|------|------|
| $z < -z_{\text{entry}}$ | 价差过低 → **做多 A, 做空 B** |
| $z > +z_{\text{entry}}$ | 价差过高 → **做空 A, 做多 B** |
| $|z| < z_{\text{exit}}$ | 价差回归 → **平仓** |

我们的默认值：$z_{\text{entry}} = 2.0$。

---
## 6 · 逐步实现：从零构建配对交易

### 6.1  创建协整合成配对

In [ ]:
np.random.seed(123)
n = 200
dates_pairs = pd.date_range(end=pd.Timestamp.now().normalize(), periods=n, freq="D")

# 创建具有已知协整关系的配对
# B 是驱动因子，A = beta*B + 均值回归噪声
BETA_TRUE = 1.5
SPREAD_MEAN = 50.0
SPREAD_STD  = 10.0

b_returns = np.random.normal(0.002, 0.03, n)
price_b = 100 * np.exp(np.cumsum(b_returns))

# 均值回归价差（Ornstein-Uhlenbeck 过程）
theta = 0.1   # 回归速度
noise_std = 3.0
spread = np.zeros(n)
spread[0] = SPREAD_MEAN
for t in range(1, n):
    spread[t] = spread[t-1] + theta * (SPREAD_MEAN - spread[t-1]) + np.random.normal(0, noise_std)

price_a = BETA_TRUE * price_b + spread

pair_closes = pd.DataFrame({
    "ASSET_A": price_a,
    "ASSET_B": price_b,
}, index=dates_pairs)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
pair_closes.plot(ax=axes[0], title="协整配对 – 价格序列")
axes[1].plot(spread, color="orange", lw=1.2)
axes[1].axhline(SPREAD_MEAN, color="gray", ls="--")
axes[1].set_title(f"真实价差 (均值={SPREAD_MEAN})")
axes[1].set_ylabel("价差")
plt.tight_layout()
plt.show()

### 6.2  OLS 对冲比率

In [ ]:
def compute_hedge_ratio(series_a: pd.Series, series_b: pd.Series) -> float:
    """OLS 回归: A = beta * B + 截距。"""
    slope, intercept, r_value, p_value, std_err = stats.linregress(
        series_b.values.astype(float),
        series_a.values.astype(float),
    )
    return float(slope)

estimated_beta = compute_hedge_ratio(pair_closes["ASSET_A"], pair_closes["ASSET_B"])
print(f"真实 β  = {BETA_TRUE:.3f}")
print(f"估计 β  = {estimated_beta:.3f}")
print(f"误差    = {abs(estimated_beta - BETA_TRUE):.4f}")

### 6.3  计算价差 & ADF 检验

In [ ]:
# 估计价差
est_spread = pair_closes["ASSET_A"] - estimated_beta * pair_closes["ASSET_B"]

# 简易 ADF：对 ΔS 关于 S_{t-1} 回归
def simple_adf_pvalue(spread: pd.Series) -> float:
    """通过 OLS 回归进行简易 ADF 检验。"""
    lagged = spread.shift(1).iloc[1:]
    delta  = spread.diff().iloc[1:]
    if lagged.std() == 0:
        return 1.0
    slope, intercept, r_value, p_value, std_err = stats.linregress(
        lagged.values.astype(float),
        delta.values.astype(float),
    )
    return float(p_value)

adf_p = simple_adf_pvalue(est_spread)
print(f"ADF p-value: {adf_p:.4f}")
print(f"在 5% 水平下协整: {'是 ✅' if adf_p < 0.05 else '否 ❌'}")

### 6.4  均值回归半衰期

从 AR(1) 模型 $\Delta S_t = \phi \cdot S_{t-1} + \varepsilon$，半衰期为：

$$
h = -\frac{\ln 2}{\phi}
$$

半衰期为 5 天意味着价差通常在 5 天内回归到均值的一半。

In [ ]:
def estimate_half_life(spread: pd.Series) -> float:
    """使用 AR(1) 估计均值回归半衰期。"""
    lagged = spread.shift(1).dropna()
    delta  = spread.diff().dropna()
    common = lagged.index.intersection(delta.index)
    if len(common) < 3:
        return float("inf")
    slope, _, _, _, _ = stats.linregress(
        lagged.loc[common].values.astype(float),
        delta.loc[common].values.astype(float),
    )
    if slope >= 0:
        return float("inf")  # 非均值回归
    return float(-np.log(2) / slope)

hl = estimate_half_life(est_spread)
print(f"估计半衰期: {hl:.1f} 天")
print(f"(真实 θ = {theta}, 理论半衰期 ≈ {-np.log(2)/np.log(1-theta):.1f} 天)")

### 6.5  Z-Score 与交易信号

In [ ]:
# Z-score
spread_mean = est_spread.mean()
spread_std  = est_spread.std()
z_score = (est_spread - spread_mean) / spread_std

Z_ENTRY = 2.0
Z_EXIT  = 0.5

fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

# ── 价差 ──
ax = axes[0]
ax.plot(est_spread.index, est_spread, lw=1.2, label="估计价差")
ax.axhline(spread_mean, color="gray", ls="--", label=f"均值 = {spread_mean:.1f}")
ax.axhline(spread_mean + Z_ENTRY * spread_std, color="red", ls=":", label=f"+{Z_ENTRY}σ")
ax.axhline(spread_mean - Z_ENTRY * spread_std, color="green", ls=":", label=f"-{Z_ENTRY}σ")
ax.set_ylabel("价差")
ax.legend(fontsize=8)
ax.set_title("价差与入场阈值")

# ── Z-Score ──
ax2 = axes[1]
ax2.plot(z_score.index, z_score, lw=1.2, color="purple")
ax2.axhline(Z_ENTRY, color="red", ls=":", label=f"入场 = ±{Z_ENTRY}")
ax2.axhline(-Z_ENTRY, color="green", ls=":")
ax2.axhline(Z_EXIT, color="orange", ls="--", alpha=0.5, label=f"出场 = ±{Z_EXIT}")
ax2.axhline(-Z_EXIT, color="orange", ls="--", alpha=0.5)
ax2.axhline(0, color="gray", lw=0.5)
ax2.fill_between(z_score.index, -Z_ENTRY, Z_ENTRY, alpha=0.05, color="gray")
ax2.set_ylabel("Z-Score")
ax2.legend(fontsize=8)
ax2.xaxis.set_major_formatter(mdates.DateFormatter("%m-%d"))

plt.tight_layout()
plt.show()

### 6.6  模拟配对交易损益

In [ ]:
# ── 简单配对交易回测 ───────────────────────────────────────
position = 0  # +1 = 做多价差, -1 = 做空价差, 0 = 空仓
pnl_list = []
trades = []

for i in range(1, len(z_score)):
    z = z_score.iloc[i]
    spread_return = est_spread.iloc[i] - est_spread.iloc[i-1]
    
    # 现有仓位的损益
    pnl = position * spread_return
    pnl_list.append({"date": z_score.index[i], "pnl": pnl, "position": position, "z": z})
    
    # 入场 / 出场逻辑
    if position == 0:
        if z < -Z_ENTRY:
            position = 1   # 做多价差（买 A, 卖 B）
            trades.append((z_score.index[i], "做多价差", z))
        elif z > Z_ENTRY:
            position = -1  # 做空价差（卖 A, 买 B）
            trades.append((z_score.index[i], "做空价差", z))
    elif position == 1 and z > -Z_EXIT:
        position = 0
        trades.append((z_score.index[i], "平多仓", z))
    elif position == -1 and z < Z_EXIT:
        position = 0
        trades.append((z_score.index[i], "平空仓", z))

pnl_df = pd.DataFrame(pnl_list).set_index("date")
pnl_df["cumulative_pnl"] = pnl_df["pnl"].cumsum()

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(pnl_df.index, pnl_df["cumulative_pnl"], lw=1.5, color="#2ecc71")
ax.set_ylabel("累计损益（价差单位）")
ax.set_title("配对交易回测 – 累计损益")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%m-%d"))
plt.tight_layout()
plt.show()

print(f"总交易次数: {len(trades)}")
print(f"最终损益: {pnl_df['cumulative_pnl'].iloc[-1]:.2f}")
for dt, action, z in trades[:10]:
    print(f"  {dt.strftime('%Y-%m-%d')}  {action:15s}  z={z:+.2f}")

---
## 7 · 生产代码：`find_cointegrated_pairs()`

我们的生产代码自动化了整个流程：筛选所有 $(N \choose 2)$ 配对，检验协整性，估计半衰期，返回评分后的 `PairSignal` 对象。

In [ ]:
from bot.signals.pairs_rotation import (
    find_cointegrated_pairs,
    pairs_rotation_weights,
    PairSignal,
)

# 创建多资产面板用于配对筛选
np.random.seed(77)
SYMBOLS_PAIR = ["BTCUSDT", "ETHUSDT", "SOLUSDT", "ADAUSDT", "XRPUSDT", "DOTUSDT"]
n_pair = 120
dates_p = pd.date_range(end=pd.Timestamp.now().normalize(), periods=n_pair, freq="D")

# 通过构造使某些配对协整
driver = np.cumsum(np.random.normal(0.001, 0.02, n_pair))
multi_closes = pd.DataFrame(index=dates_p)
multi_closes["BTCUSDT"] = 60000 * np.exp(driver)
multi_closes["ETHUSDT"] = 3500 * np.exp(driver * 1.3 + np.cumsum(np.random.normal(0, 0.005, n_pair)))
multi_closes["SOLUSDT"] = 140 * np.exp(np.cumsum(np.random.normal(0.003, 0.04, n_pair)))
multi_closes["ADAUSDT"] = 0.45 * np.exp(np.cumsum(np.random.normal(-0.001, 0.03, n_pair)))
multi_closes["XRPUSDT"] = 0.55 * np.exp(driver * 0.8 + np.cumsum(np.random.normal(0, 0.008, n_pair)))
multi_closes["DOTUSDT"] = 7 * np.exp(np.cumsum(np.random.normal(0.002, 0.035, n_pair)))

# 查找协整配对
pair_signals = find_cointegrated_pairs(
    multi_closes,
    lookback=60,
    adf_threshold=0.05,
    min_half_life=1.0,
    max_half_life=30.0,
)

print(f"发现 {len(pair_signals)} 个协整配对：\n")
for ps in pair_signals:
    print(f"  {ps.asset_a} / {ps.asset_b}")
    print(f"    z-score={ps.z_score:+.2f}  对冲比率={ps.hedge_ratio:.4f}")
    print(f"    半衰期={ps.half_life:.1f}天  ADF p={ps.adf_pvalue:.4f}")
    print()

In [ ]:
# ── 将配对信号转换为投资组合权重 ───────────────────────────
weights = pairs_rotation_weights(pair_signals, z_entry=2.0, max_pairs=3)

print("配对轮换权重调整:")
for sym, w in sorted(weights.items(), key=lambda x: x[1], reverse=True):
    print(f"  {sym:10s}  {w:+.3f}")

if not weights:
    print("  (无可操作配对 — 所有 z-score 在 ±2.0 以内)")

---
## 8 · `PairSignal` 解剖

```python
@dataclass(frozen=True, slots=True)
class PairSignal:
    asset_a: str         # 配对中的第一个资产
    asset_b: str         # 第二个资产
    z_score: float       # 当前标准化价差
    spread: float        # 原始价差值
    half_life: float     # 估计回归 50% 所需天数
    hedge_ratio: float   # 价差的 OLS β
    adf_pvalue: float    # ADF 检验 p 值 (< 0.05 = 协整)
```

**关键设计选择：**
- 按 `|z_score|` 降序排列 — 最极端的配对排在最前。
- `pairs_rotation_weights()` 将信号转换为投资组合权重*调整量*（空头腿可以为负）。
- `max_pairs=3` 上限防止过度集中。

---
## 9 · 均值回归 vs 动量：何时使用哪个？

| 因素 | 动量 | 均值回归 |
|------|------|----------|
| **市场制度** | 强趋势（牛/熊） | 震荡 / 区间波动 |
| **时间跨度** | 中期（天-周） | 短期（小时-天） |
| **风险特征** | 动量崩溃 | 接飞刀 |
| **最佳搭配** | 趋势确认（EMA） | 协整检验 |

这正是为什么我们的机器人使用**制度自适应集成** — 制度检测器（NB06）决定哪个策略获得更多权重。

在**牛市**制度下：动量权重 ↑，均值回归权重 ↓  
在**震荡**制度下：动量权重 ↓，均值回归权重 ↑

---
## 10 · 关键要点

| 概念 | 详情 |
|------|------|
| **信号强度** | max(RSI_signal, BB_signal) ∈ [0, 1] |
| **协整** | 两个序列由平稳价差绑定 (ADF p < 0.05) |
| **对冲比率** | OLS 斜率 β，使得 A - βB 平稳 |
| **半衰期** | 基于 AR(1) 的回归速度估计 |
| **Z-score 入场** | |z| > 2.0 → 交易; |z| < 0.5 → 出场 |
| **配置** | `strategy_params.yaml` → `mean_reversion:` 部分 |

---
## 🔬 练习

1. **强度分解：** 将信号修改为使用 RSI 和 BB 信号的*加权平均*而非 `max()`。`0.6 * rsi_signal + 0.4 * bb_signal` 是否产生更一致的入场点？

2. **半衰期过滤：** 在配对回测中，跳过半衰期 > 10 天的配对。这是否改善了损益？

3. **滚动协整：** 协整关系可能随时间崩解。实现一个滚动 60 天窗口的 ADF 检验，绘制 BTCUSDT / ETHUSDT 配对的 p 值随时间变化。

4. **配对止损：** 在配对回测中添加止损：入场后若 z-score 反向移动 1.5 倍（例如在 z = -2 入场，z 达到 -5 时退出），则平仓。这是否减少了回撤？

---
## ✅ 知识检查

1. *相关性*和*协整性*有什么区别？
2. 半衰期为 7 天的直觉含义是什么？
3. 为什么生产代码使用 `max(RSI_signal, BB_signal)` 而不是将它们相加？
4. 如果 ADF p 值为 0.50 会怎样？你应该交易那个配对吗？
5. `pairs_rotation_weights()` 如何处理 z-score 为 -3.0（即 z < -z_entry）的情况？

---
## 🔗 下一步

**[NB06 — 制度检测 →](06_制度检测.ipynb)**

我们将学习机器人如何将市场分类为**牛市 / 震荡 / 熊市**制度 — 这是决定*哪个*策略获得最多权重的关键。